# Basile engine 1 · The recipe, and a toy library checked exhaustively
**Engine:** `ValaQuenta.modules.basile` · confidence floor ESTABLISHED.

The Library of Babel family joins a *location* (a number) to a *page* (a numeral) by an invertible pseudo-random map, so nothing is stored. Basile's published README gives the recipe: an LCG `(a·p + c) mod m`, then shift/XOR steps; inverse = undo the steps in reverse, then `a⁻¹(p − c) mod m`. This notebook exercises the **generic** recipe (written from the README, no published code or constant used) on a library small enough to check every location.

In [1]:
import sys, os, importlib.util, math, time, random
sys.path.insert(0, os.path.abspath('../..'))
def load_maths():
    spec = importlib.util.spec_from_file_location("basile_maths", "../../modules/basile/maths.py")
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
M = load_maths()
CACHE = M.default_cache_dir()
HAVE = os.path.exists(os.path.join(CACHE, "PRNG")) and os.path.exists(os.path.join(CACHE, "imagesearch.cpp"))
print("engine loaded; published files in cache:", HAVE, "| gmpy2 accelerator:", M._gmpy2 is not None)

engine loaded; published files in cache: True | gmpy2 accelerator: True


## The ordered address: the same set of pages in rank order
Bijective base-N (shortest first). `ordered_address` and `ordered_text_at` are inverses.

In [2]:
A = "abcdefghijklmnopqrstuvwxyz"
print([(n, M.ordered_text_at(n, A)) for n in (0, 25, 26, 701, 702)])
assert all(M.ordered_address(M.ordered_text_at(n, A), A) == n for n in range(20000))
print("round trip exact for 20,000 addresses")

[(0, 'a'), (25, 'z'), (26, 'aa'), (701, 'zz'), (702, 'aaa')]
round trip exact for 20,000 addresses


## The two xorshift steps have exact inverses
Right xorshift: `y ⊕ (y≫s) ⊕ (y≫2s) ⊕ …`. Masked left xorshift: ⌈k/s⌉ fixed-point iterations, each fixing `s` more low bits.

In [3]:
rnd = random.Random(1)
for width, s in ((64, 7), (257, 11), (1000, 13), (5000, 1)):
    for _ in range(300):
        x = rnd.getrandbits(width); k = width - s
        assert M.xorshift_right_inverse(M.xorshift_right(x, s), s) == x
        assert M.xorshift_left_masked_inverse(M.xorshift_left_masked(x, k, s), k, s) == x
print("both inverses exact on 1,200 random words at four widths")

both inverses exact on 1,200 random words at four widths


## The scrambler of [0, N): LCG modulo 2^e + two xorshifts + cycle-walking
N = 29³ = 24,389 pages (29 symbols, 3-character pages). **Every** location is checked.

In [4]:
N = 29 ** 3
sc, un = M.power_of_two_scrambler(N)
vals = [sc(x) for x in range(N)]
assert sorted(vals) == list(range(N)) and all(un(sc(x)) == x for x in range(N))
print("N =", N, "| a bijection on [0, N): True | every location round-trips: True")
print("Hull-Dobell (m = 2^15, a = 1664525, c = 1013904223, prime factors [2]):", M.hull_dobell(1 << 15, 1664525, 1013904223, [2]))

N = 24389 | a bijection on [0, N): True | every location round-trips: True
Hull-Dobell (m = 2^15, a = 1664525, c = 1013904223, prime factors [2]): {'gcd_m_c_is_1': True, 'a_minus_1_divisible_by_prime_factors': True, 'a_minus_1_divisible_by_4_if_4_divides_m': True, 'full_period': True}


## A page and its location, both directions

In [5]:
print("loc   scrambled  ordered")
for loc in range(8): print("%-5d %-10r %r" % (loc, M.babel_page(loc), M.page_at(loc, M.BABEL_ALPHABET, 3)))
assert all(M.babel_location(M.babel_page(l)) == l for l in (0, 1, 777, N - 1))

loc   scrambled  ordered
0     'blr'      'aaa'
1     'lrq'      'aab'
2     'wfl'      'aac'
3     'is,'      'aad'
4     'did'      'aae'
5     ',dz'      'aaf'
6     'qxx'      'aag'
7     'bw '      'aah'


## Neighbouring locations
Mean number of the 3 characters that differ between the pages at L and L+1, over all 24,388 pairs.

In [6]:
d = lambda a, b: sum(x != y for x, y in zip(a, b))
o = sum(d(M.page_at(i, M.BABEL_ALPHABET, 3), M.page_at(i + 1, M.BABEL_ALPHABET, 3)) for i in range(N - 1)) / (N - 1)
s = sum(d(M.page_at(sc(i), M.BABEL_ALPHABET, 3), M.page_at(sc(i + 1), M.BABEL_ALPHABET, 3)) for i in range(N - 1)) / (N - 1)
print("ordered %.3f | scrambled %.3f | independent pages would give %.3f" % (o, s, 3 * 28 / 29))

ordered 1.036 | scrambled 2.910 | independent pages would give 2.897


## Cycle structure: which part is cyclic
The LCG modulo 2^e is one full cycle (Hull–Dobell). The scrambler built on it is a permutation with several cycles; its spectrum is the roots of unity of each cycle length present.

In [7]:
c = M.permutation_cycles(sc, N)
big = 1 << 15
lcg = M.permutation_cycles(lambda x: (1664525 * x + 1013904223) % big, big)
xs = M.permutation_cycles(lambda x: M.xorshift_left_masked(M.xorshift_right(x, 7), 10, 5) & (big - 1), big)
print("pure LCG mod 2^15  : cycles", len(lcg), "| longest", lcg[0])
print("scrambler (N=%d)   : cycles %d | longest %d | fixed points %d | top lengths %s" % (N, len(c), c[0], c.count(1), c[:8]))
assert lcg == [big] and len(c) > 1

pure LCG mod 2^15  : cycles 1 | longest 32768
scrambler (N=24389)   : cycles 11 | longest 9943 | fixed points 1 | top lengths [9943, 6700, 6387, 770, 479, 34, 32, 28]


## Open items, stated
* This is **our** realisation of the recipe. It is not the text library's algorithm, which is not published (Basile's README: the published one is the more efficient rewrite for the image library).
* Bijectivity here comes from a power-of-two modulus (Hull–Dobell for `2^e`) and cycle-walking. The published Babelia code uses a different modulus (a 3.19-million-bit `m`, not a power of two); notebook 2 audits it.